In [1]:
from google.adk.agents import Agent
from google.adk.agents import LlmAgent
from google.adk.tools.google_search_tool import GoogleSearchTool

In [2]:
import vertexai

# Initialize AI Platform
PROJECT_ID = "qwiklabs-gcp-04-a9b50dbd3df9"
LOCATION = "us-central1"
STAGING_BUCKET = "gs://vertex_ai_genai"

vertexai.init(
  project=PROJECT_ID,
  location=LOCATION,
  staging_bucket=STAGING_BUCKET,
)

In [3]:
# Critique Agent - Reviews search results and suggests improvements
critique_agent = Agent(
    model="gemini-2.5-flash",
    name='critique_agent',
    description="Reviews responses and provides constructive feedback for improvement.",
    instruction="""You are a critique agent. You will receive a search response and must analyze it critically.

Your job is to:
1. Evaluate the completeness of the answer
2. Identify any missing information or unclear explanations
3. Suggest specific improvements

Output a structured critique with:
- STRENGTHS: What's good about the response
- WEAKNESSES: What's missing or could be better
- SUGGESTIONS: Specific recommendations for improvement

Be constructive and specific in your feedback.""",
)

# Refine Agent - Rewrites the response based on critique
refine_agent = Agent(
    model="gemini-2.5-flash",
    name='refine_agent',
    description="Rewrites and improves responses based on critique feedback.",
    instruction="""You are a refine agent. You will receive:
1. An original search response
2. A critique with suggestions for improvement

Your job is to rewrite the response incorporating the feedback from the critique.
Make the response:
- More complete and informative
- Clearer and better organized
- Address the weaknesses identified in the critique

Output ONLY the refined, improved response - no meta-commentary about your changes.""",
)

search_agent = LlmAgent(
    model="gemini-2.5-flash",  # A fast model suitable for search tasks
    name="SearchAgent",
    instruction="""You are an agent with the autonomy to research on the web.
    Search the requested topics using the 'google_search' tool and provide a summary of the findings.
    """,
    tools=[GoogleSearchTool()], # Add the built-in GoogleSearchTool
)

In [4]:
def append_to_state(tool_context, field, response):
  existing_state = tool_context.state.get(field, [])
  tool_context.state[field] = existing_state + [response]
  return {"status": "success"}

In [5]:
from google.adk.agents import LoopAgent

writers_room = LoopAgent(
  name="writers_room",
  description="Iterates ",
  sub_agents=[
  search_agent,
  critique_agent,
  refine_agent
  ],
  max_iterations=3,
)

In [6]:
root_agent = Agent(
  name="greeter",
  model="gemini-2.5-flash",
  description="You are helpful assistant.",
  instruction="""
    You are a helpful assistant that routes user requests to the appropriate specialist agent.
    For weather-related questions (forecasts, temperature, climate, etc.), delegate to weather_agent.
    For all other questions (news, general knowledge, searches), delegate to search_agent.
    Always delegate to the appropriate agent - do not try to answer directly.
  """,
  tools=[append_to_state],
  sub_agents=[writers_room],
)

In [7]:
from vertexai.preview import reasoning_engines
app = reasoning_engines.AdkApp(
  agent=root_agent,
  enable_tracing=False,
)

In [8]:
user_id = "test-user-id"
session = app.create_session(user_id=user_id)
print(session.id)

This legacy setting overrides the new Cloud Console toggle and environment variable controls.
Impact: The Cloud Console may incorrectly show telemetry as 'On' when it is actually 'Off', and the UI toggle will not work.
Action: To fix this and control telemetry, please remove the 'enable_tracing' parameter from your deployment code.
You can then use the 'GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY' environment variable:
agent_engines.create(
  env_vars={
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": true|false
  }
)
or the toggle in the Cloud Console: https://console.cloud.google.com/vertex-ai/agents.


060f200c-f62c-4397-88a8-f723e864d6a5


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:825: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


In [9]:
from IPython.display import Markdown, display
for event in app.stream_query(
  user_id=user_id,
  session_id=session.id,
  message="Explain Quantum computing",
):
  lastevent = event
display(Markdown(lastevent["content"]["parts"][0]["text"]))

Quantum computing is a new type of computing that harnesses the principles of quantum mechanics to perform calculations. Unlike classical computers that use bits to represent information as either a 0 or a 1, quantum computers use **qubits**.

Here's a breakdown of the core concepts:

1.  **Qubits (Quantum Bits):**
    *   A qubit can represent a 0, a 1, or both at the same time. This "both at the same time" state is called **superposition**.
    *   Imagine a coin spinning in the air; it's neither heads nor tails until it lands. A qubit in superposition is similar – it exists in a combination of possibilities simultaneously. This allows quantum computers to process multiple possibilities concurrently.

2.  **Superposition:**
    *   This is the ability of a quantum system to be in multiple states at once. For example, an electron can spin up and spin down simultaneously.
    *   This property allows quantum computers to explore many computational paths in parallel, leading to potential speedups for certain problems.

3.  **Entanglement:**
    *   When qubits become "entangled," they are linked in such a way that the state of one instantly influences the state of the other, no matter how far apart they are.
    *   This interconnectedness allows quantum computers to perform highly complex calculations on related data points in a way classical computers cannot.

4.  **Quantum Interference:**
    *   Just like waves can interfere with each other (constructively or destructively), quantum states can interfere.
    *   Quantum algorithms are designed to amplify the probability of correct answers through constructive interference and suppress incorrect answers through destructive interference.

**How it Works (Simplified):**
Quantum computers manipulate qubits using "quantum gates" (analogous to logic gates in classical computers). These gates perform operations that change the quantum state of the qubits. Because of superposition and entanglement, a quantum computer can hold and process vast amounts of information simultaneously compared to a classical computer.

**Potential Applications:**
Quantum computing is still in its early stages, but it promises to revolutionize fields such as:
*   **Drug Discovery and Materials Science:** Simulating molecular structures and reactions with unprecedented accuracy to design new medicines and materials.
*   **Cryptography:** Developing new, unhackable encryption methods and potentially breaking current ones.
*   **Artificial Intelligence:** Enhancing machine learning algorithms and solving complex optimization problems.
*   **Financial Modeling:** Creating more accurate and complex financial models.

**Challenges:**
Building and maintaining quantum computers is extremely difficult. Qubits are very sensitive to environmental noise (like temperature fluctuations or electromagnetic fields), which can cause them to lose their quantum properties (a process called decoherence). This requires maintaining qubits in extremely cold, controlled environments.

In essence, quantum computing leverages the peculiar and powerful rules of the quantum world to tackle problems that are beyond the capabilities of even the most powerful classical supercomputers.

In [10]:
from vertexai import agent_engines

remote_agent = agent_engines.create(
  app,
  requirements=["google-cloud-aiplatform[agent_engines,adk]"],
)

INFO:vertexai.agent_engines:Identified the following requirements: {'google-cloud-aiplatform': '1.135.0', 'cloudpickle': '3.1.2', 'pydantic': '2.12.3'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.12.3'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'cloudpickle==3.1.2', 'pydantic==2.12.3']
INFO:vertexai.agent_engines:Using bucket vertex_ai_genai
INFO:vertexai.agent_engines:Wrote to gs://vertex_ai_genai/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://vertex_ai_genai/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://vertex_ai_genai/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/1717239328/locations/us-central1/reasoningEngines/815919321523735756

In [13]:
for event in remote_agent.stream_query(
  user_id="agent-engine-test-user",
  message="Explain Quantum computing",
):
  print(event)

{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'text': 'Quantum computing is a new paradigm of computing that leverages the principles of quantum mechanics, such as superposition and entanglement, to perform calculations.\n\nHere\'s a breakdown of its core concepts:\n\n1.  **Qubits:** Unlike classical computers that use bits to represent information as either a 0 or a 1, quantum computers use "qubits." A qubit can represent 0, 1, or a combination of both simultaneously through a phenomenon called **superposition**. This means a single qubit can hold more information than a classical bit.\n\n2.  **Superposition:** Imagine a coin spinning in the air. While it\'s spinning, it\'s neither heads nor tails, but a combination of both possibilities. This is analogous to a qubit in superposition, existing in multiple states at once until it\'s measured.\n\n3.  **Entanglement:** This is a unique quantum phenomenon where two or more qubits become linked in such a way that they share 